# ConvNeXt model

U ovoj svesci treniramo ConvNeXt-Tiny model za klasifikaciju slika na Tiny ImageNet skupu podataka, koji se sastoji od 200 klasa. Cilj ovog eksperimenta je da dobijemo osnovne performanse ConvNeXt-Tiny modela.Tokom treninga prate se train loss, validation loss, train accuracy i validation accuracy, a najbolji model se čuva na osnovu validation accuracy rezultata.

In [ ]:
from google.colab import files
uploaded = files.upload()  # opens a file picker, select subset.zip

Saving tiny-imagenet-200-modified.zip to tiny-imagenet-200-modified.zip


In [ ]:
!unzip -q tiny-imagenet-200-modified.zip -d /content/tiny-imagenet-200-modified

Učitavanje biblioteka

In [ ]:
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import torchvision.transforms.v2 as v2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import Subset
from torchvision import datasets, transforms
from torchvision.models import (
    convnext_tiny,
    ConvNeXt_Tiny_Weights)
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix)
from PIL import Image
import random
import torch.nn.functional as F
from pathlib import Path


Postavljanje početne vrednosti generatora slučajnih brojeva kako bi rezultati treniranja bili što ponovljiviji pri ponovnom pokretanju koda.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Izbor uređaja na kojem će se model izvršavati. Ako je dostupna CUDA podrška, koristi se GPU, a u suprotnom se model izvršava na CPU-u. Funkcija bind_gpu omogućava prebacivanje podataka na izabrani uređaj

In [ ]:
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def bind_gpu(data):
    device = get_device()
    if isinstance(data, (list, tuple)):
        return [bind_gpu(data_elem) for data_elem in data]
    else:
        return data.to(device, non_blocking=True)
device=get_device()

Desifinisanje osnovnih parametara za treniranje modela. Originalna veličina slika u Tiny ImageNet skupu je 64x64, dok smo u projektu koristili veličinu slika 224×224. Veličina batch-a je postavljena na 256, dok je broj epoha 100


In [ ]:
MODEL_NAME = "ConvNeXt_224"
DATA_LOCATION = Path("/content/tiny-imagenet-200-modified/tiny-imagenet-200-modified")
OUTPUT_LOCATION = Path("/content/runs") / MODEL_NAME
NUM_CLASSES = 200
IMG_SIZE = 224
BATCH_SIZE = 256
NUM_WORKERS = 8
EPOCHS = 100
LR = 0.0001
WEIGHT_DECAY = 0.05

Definisanje transformacije koje se primenjuju na slike tokom treniranja i validacije. Veličina izlaznih slika određena je promenljivom IMG_SIZE, pa se isti kod može koristiti za slike veličine 64×64 ili 224×224. Tokom treniranja koriste se nasumično isecanje i horizontalno okretanje kako bi se povećala raznovrsnost podataka i smanjio rizik od overfitting-a.Tokom validacije slika se samo prilagođava potrebnoj veličini, bez dodatnih nasumičnih promena. Na kraju se slike pretvaraju u tenzore i normalizuju pomoću standardnih ImageNet vrednosti.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=27),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
    transforms.RandomErasing(p=0.25)
])


val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

Učitavanje modifikovanog Tiny ImageNet skupa podataka koji je korišćen zbog smanjenja vremena izvršavanja programa. Podaci su organizovani u 200 klasa, a posebno se učitavaju skupovi za trening i validaciju. Nakon toga se formiraju DataLoader objekti koji podatke dele u batch-eve i prosleđuju ih modelu tokom treninga i validacije. 

In [ ]:
train_dataset = datasets.ImageFolder(DATA_LOCATION / "train", transform = train_transform)
val_dataset = datasets.ImageFolder(DATA_LOCATION / "val", transform = val_transform)

print()
print("Train images:",len(train_dataset))
print("Validation images:",len(val_dataset))


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)

print()
print("Train batches:",len(train_loader))
print("Validation batches:",len(val_loader))





Train images: 95000
Validation images: 10000

Train batches: 371
Validation batches: 40


Definicija ConvNeXt-Tiny modela koji se trenira od početka, bez korišćenja prethodno istreniranih težina (weights=None). Završni klasifikacioni sloj prilagođen je broju klasa u skupu podataka. Zatim se definišu funkcija gubitka CrossEntropyLoss, optimizer AdamW i scheduler CosineAnnealingLR za postepeno prilagođavanje learning rate-a tokom treniranja. Na kraju se ispisuje ukupan broj parametara modela.

In [ ]:
class ConvNeXtClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = convnext_tiny(weights=None)
        in_features = (self.model.classifier[2].in_features)
        self.model.classifier[2] = nn.Linear(in_features,num_classes)

    def forward(self,x):
        return self.model(x)

model = ConvNeXtClassifier(NUM_CLASSES).to(device)
print(model)

parameters = sum(p.numel() for p in model.parameters())
print(f"Parameters: {parameters/1e6:.2f}M")
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)

mixup = v2.MixUp(alpha=0.8, num_classes=NUM_CLASSES)
cutmix = v2.CutMix(alpha=1.0, num_classes=NUM_CLASSES)
mixcut = v2.RandomChoice([mixup, cutmix])

ConvNeXtClassifier(
  (model): ConvNeXt(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
        (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
      )
      (1): Sequential(
        (0): CNBlock(
          (block): Sequential(
            (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
            (1): Permute()
            (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
            (3): Linear(in_features=96, out_features=384, bias=True)
            (4): GELU(approximate='none')
            (5): Linear(in_features=384, out_features=96, bias=True)
            (6): Permute()
          )
          (stochastic_depth): StochasticDepth(p=0.0, mode=row)
        )
        (1): CNBlock(
          (block): Sequential(
            (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
            (1): Permute()
            (2): LayerNorm(

Ova ćelija sadrži proces treniranja i validacije ConvNeXt-Tiny modela. Tokom svake epohe model se trenira na trening skupu, a zatim se proverava na validacionom skupu. Prate se vrednosti loss-a i accuracy-ja, a najbolji model se čuva na osnovu najbolje validation accuracy vrednosti. Tokom treniranja koristi se progress bar kako bi se pratilo izvršavanje. Takođe se čuvaju metrike potrebne za kasniju analizu i prikaz rezultata.

In [ ]:
metrics = {
    "epoch": [],
    "training_loss": [],
    "training_accuracy": [],
    "validation_loss": [],
    "validation_accuracy": [],
    "current_lr": [],
    "epoch_time": [],
}

OUTPUT_LOCATION.mkdir(parents=True, exist_ok=True)

best_val_accuracy = 0.0
best_epoch = 0

best_model_path = OUTPUT_LOCATION / "convnext_best_model.pth"

pbar = tqdm(
    total=EPOCHS,
    desc="Training Progress"
)
pbar.set_postfix({
    "loss": -1,
    "accuracy": -1
})

for epoch in range(EPOCHS):
    start_time = time.time()

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_samples = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        images, labels = mixcut(images, labels)

        optimizer.zero_grad()

        with torch.autocast(device_type=device.type,dtype=torch.bfloat16,enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()


        predicted = outputs.argmax(dim=1)


        target_classes = labels.argmax(dim=1)

        train_correct += (predicted == target_classes).sum().item()

        train_samples += images.size(0)

        train_loss += loss.item() * images.size(0)

    train_loss /= train_samples
    train_accuracy = train_correct / train_samples

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_samples = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            with torch.autocast(device_type=device.type,dtype=torch.bfloat16,enabled=(device.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            predicted = outputs.argmax(dim=1)

            val_correct += (predicted == labels).sum().item()

            val_samples += labels.size(0)

    val_loss /= val_samples
    val_accuracy = val_correct / val_samples


    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_epoch = epoch + 1
        torch.save(model.state_dict(),best_model_path)

        print(f"Saved best model to: {best_model_path}")

    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    epoch_train_time = time.time() - start_time

    metrics["training_loss"].append(train_loss)
    metrics["training_accuracy"].append(train_accuracy)
    metrics["validation_loss"].append(val_loss)
    metrics["validation_accuracy"].append(val_accuracy)
    metrics["current_lr"].append(current_lr)
    metrics["epoch_time"].append(epoch_train_time)
    metrics["epoch"].append(epoch + 1)


    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Train loss: {train_loss:.4f}, "
        f"Train acc: {train_accuracy:.4f} | "
        f"Val loss: {val_loss:.4f}, "
        f"Val acc: {val_accuracy:.4f}"
    )

    pbar.set_postfix({"loss": f"{val_loss:.4f}","accuracy": f"{val_accuracy:.4f}"})

    pbar.update(1)

pbar.close()


history_path = OUTPUT_LOCATION / "metrics_cn.csv"

df = pd.DataFrame(metrics)
df.to_csv(history_path, index=False)



Training Progress:   1%|          | 1/100 [01:23<2:17:21, 83.25s/it, loss=5.0533, accuracy=0.0289]

Saved best model to: /content/runs/ConvNeXt_224/convnext_best_model.pth
Epoch [1/100] Train loss: 5.2602, Train acc: 0.0129 | Val loss: 5.0533, Val acc: 0.0289


Training Progress:   2%|▏         | 2/100 [02:46<2:15:52, 83.18s/it, loss=4.9427, accuracy=0.0418]

Saved best model to: /content/runs/ConvNeXt_224/convnext_best_model.pth
Epoch [2/100] Train loss: 5.2134, Train acc: 0.0184 | Val loss: 4.9427, Val acc: 0.0418


Training Progress:   3%|▎         | 3/100 [04:09<2:14:38, 83.28s/it, loss=4.8723, accuracy=0.0530]

...

Saved best model to: /content/runs/ConvNeXt_224/convnext_best_model.pth
Epoch [98/100] Train loss: 3.9450, Train acc: 0.2996 | Val loss: 2.7240, Val acc: 0.4945


Training Progress:  99%|█████████▉| 99/100 [2:17:27<01:23, 83.36s/it, loss=2.7235, accuracy=0.4941]

Epoch [99/100] Train loss: 3.9151, Train acc: 0.3010 | Val loss: 2.7235, Val acc: 0.4941


Training Progress: 100%|██████████| 100/100 [2:18:51<00:00, 83.31s/it, loss=2.7234, accuracy=0.4942]

Epoch [100/100] Train loss: 3.9455, Train acc: 0.2980 | Val loss: 2.7234, Val acc: 0.4942
